### Step 2) Participation in Ancillary Service Markets

### Problem Description and Scenario Generation
In Lecture 10, we explore the offering strategy for a price-taking stochastic flexible load with consumption between 0 and 600 kW. We focus on the DK2 FCR-D UP market with hourly bids. The flexible load can provide upward frequency containment by reducing consumption quickly.


We consider a single bidding hour, modeled with minute-level resolution.

Data Generation for Future Stochastic Load


Randomly generate 300 load profiles such that:

• consumption remains between 220 and 600 kW,

• minute-to-minute changes do not exceed 35 kW.


From these profiles:

• 100 are used as in-sample profiles (each equally likely),

• 200 are used as out-of-sample profiles.

### Generating Load profiles

In [1]:
import numpy as np
import pandas as pd

In [2]:
# Parameters

N_PROFILES = 300
N_MINUTES = 60
LOAD_MIN = 220
LOAD_MAX = 600
MAX_STEP = 35
N_IN_SAMPLE = 100
N_OUT_SAMPLE = 200
RANDOM_SEED = 42


In [3]:
# Profile generator

def generate_load_profile(
    n_minutes=60,
    load_min=220,
    load_max=600,
    max_step=35,
    rng=None
):
    if rng is None:
        rng = np.random.default_rng()

    profile = np.zeros(n_minutes)

    # Random starting point within allowed range
    profile[0] = rng.uniform(load_min, load_max)

    for t in range(1, n_minutes):
        prev = profile[t - 1]
        
        # Keep next value within both the global bounds and max-step bound
        lower = max(load_min, prev - max_step)
        upper = min(load_max, prev + max_step)

        profile[t] = rng.uniform(lower, upper)

    return profile

In [4]:
def generate_profiles(
    n_profiles=300,
    n_minutes=60,
    load_min=220,
    load_max=600,
    max_step=35,
    seed=42
):
    rng = np.random.default_rng(seed)

    profiles = np.array([
        generate_load_profile(
            n_minutes=n_minutes,
            load_min=load_min,
            load_max=load_max,
            max_step=max_step,
            rng=rng
        )
        for _ in range(n_profiles)
    ])

    return profiles

In [5]:
# Validation

def validate_profiles(profiles, load_min=220, load_max=600, max_step=35):
    within_bounds = np.all((profiles >= load_min) & (profiles <= load_max))
    step_ok = np.all(np.abs(np.diff(profiles, axis=1)) <= max_step)

    return within_bounds, step_ok

In [6]:
# Main

if __name__ == "__main__":
    profiles = generate_profiles(
        n_profiles=N_PROFILES,
        n_minutes=N_MINUTES,
        load_min=LOAD_MIN,
        load_max=LOAD_MAX,
        max_step=MAX_STEP,
        seed=RANDOM_SEED
    )

    within_bounds, step_ok = validate_profiles(
        profiles,
        load_min=LOAD_MIN,
        load_max=LOAD_MAX,
        max_step=MAX_STEP
    )

    print(f"Profiles shape: {profiles.shape}")
    print(f"All values within bounds: {within_bounds}")
    print(f"All minute-to-minute changes within limit: {step_ok}")

    # Split into in-sample and out-of-sample
    in_sample_profiles = profiles[:N_IN_SAMPLE]
    out_of_sample_profiles = profiles[N_IN_SAMPLE:N_IN_SAMPLE + N_OUT_SAMPLE]

    # Equal probabilities for in-sample scenarios
    in_sample_probabilities = np.full(N_IN_SAMPLE, 1 / N_IN_SAMPLE)

    print(f"In-sample shape: {in_sample_profiles.shape}")
    print(f"Out-of-sample shape: {out_of_sample_profiles.shape}")
    print(f"In-sample probabilities sum: {in_sample_probabilities.sum()}")

    # Save to CSV
    minute_cols = [f"minute_{t+1}" for t in range(N_MINUTES)]

    df_all = pd.DataFrame(profiles, columns=minute_cols)
    df_all.insert(0, "profile_id", range(1, N_PROFILES + 1))

    df_in = pd.DataFrame(in_sample_profiles, columns=minute_cols)
    df_in.insert(0, "profile_id", range(1, N_IN_SAMPLE + 1))
    df_in["probability"] = in_sample_probabilities

    df_out = pd.DataFrame(out_of_sample_profiles, columns=minute_cols)
    df_out.insert(0, "profile_id", range(N_IN_SAMPLE + 1, N_PROFILES + 1))

    df_all.to_csv("all_profiles.csv", index=False)
    df_in.to_csv("in_sample_profiles.csv", index=False)
    df_out.to_csv("out_of_sample_profiles.csv", index=False)

    print("Saved:")
    print(" - all_profiles.csv")
    print(" - in_sample_profiles.csv")
    print(" - out_of_sample_profiles.csv")

Profiles shape: (300, 60)
All values within bounds: True
All minute-to-minute changes within limit: True
In-sample shape: (100, 60)
Out-of-sample shape: (200, 60)
In-sample probabilities sum: 0.9999999999999999
Saved:
 - all_profiles.csv
 - in_sample_profiles.csv
 - out_of_sample_profiles.csv
